# 🧠 Data-Centric AI Agent — Training Notebook

**OpenEnv Hackathon 2026**

Trains **Qwen2.5-1.5B-Instruct** (4-bit QLoRA) to orchestrate data-cleaning specialists
via GRPO reinforcement learning on the Data-Centric AI environment.

| | |
|---|---|
| **Space** | https://huggingface.co/spaces/Aswini-Kumar/data-centric-env |
| **Repo** | https://github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env |
| **Algorithm** | SFT warmup → GRPO (TRL) |
| **Model** | Qwen/Qwen2.5-1.5B-Instruct, 4-bit QLoRA r=8 |

**Pipeline:**
1. Install deps
2. Clone repo + start env server
3. Generate SFT warmup data
4. Phase 1 — SFT warmup (~15 min)
5. Phase 2 — GRPO training (~45-90 min depending on hardware)
6. Plot reward curves
7. Evaluate vs baselines
8. Save model


## 📦 Step 1 — Install Dependencies

In [ ]:
# Pin transformers first to avoid torchao conflict
# (transformers ≥ 4.52 requires torchao ≥ 0.9 which needs torch.int1 — not in Colab yet)
!pip install "transformers>=4.44.0,<4.52.0" --quiet

# Unsloth — detects the installed torch version automatically
!pip install "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git" --quiet

# Training stack
!pip install trl>=0.15.0 datasets>=2.0.0 accelerate>=0.30.0 --quiet
!pip install scikit-learn>=1.3.0 pandas>=2.0.0 numpy matplotlib --quiet
!pip install "openenv-core[core]>=0.2.1" --quiet

# Verify Unsloth
from unsloth import FastLanguageModel
import torch
print(f'✅ Unsloth ready | torch {torch.__version__} | GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')


## 🔁 Step 2 — Clone Repo & Start Environment Server

In [ ]:
import os, sys, shutil, subprocess, time, requests

REPO     = 'https://github.com/CelestialWorthyOfHeavenAndEarth/data-centric-env.git'
WORK_DIR = '/content/data-centric-env'

# Clone or update
if os.path.exists(f'{WORK_DIR}/pyproject.toml'):
    print('Repo exists — pulling latest...')
    os.system(f'git -C {WORK_DIR} pull origin main')
else:
    if os.path.exists(WORK_DIR): shutil.rmtree(WORK_DIR)
    os.system(f'git clone {REPO} {WORK_DIR}')

os.chdir(WORK_DIR)
sys.path.insert(0, WORK_DIR)
os.makedirs('logs', exist_ok=True)
os.makedirs('plots', exist_ok=True)
print('CWD:', os.getcwd())
os.system('git log --oneline -3')
os.system('pip install -e . --quiet 2>/dev/null || true')
print('✅ Repo ready')


In [ ]:
import subprocess, time, requests, os

# Kill any leftover server
os.system('fuser -k 8000/tcp 2>/dev/null || true')
time.sleep(2)

server = subprocess.Popen(
    ['python', '-m', 'uvicorn', 'server.app:app',
     '--host', '0.0.0.0', '--port', '8000', '--log-level', 'warning'],
    stdout=subprocess.PIPE, stderr=subprocess.PIPE
)

for i in range(40):
    try:
        r = requests.get('http://localhost:8000/health', timeout=2)
        if r.status_code == 200:
            print(f'✅ Server ready after {i+1}s — {r.json()}')
            break
    except: time.sleep(1)
else:
    out, err = server.communicate(timeout=5)
    print('STDOUT:', out.decode()[-2000:])
    print('STDERR:', err.decode()[-2000:])
    raise RuntimeError('Server failed to start')

os.environ['ENV_URL'] = 'http://localhost:8000'
print('ENV_URL set to http://localhost:8000')


## 📊 Step 3 — Generate SFT Warmup Data

In [ ]:
import os, time
os.chdir('/content/data-centric-env')

sft_path = 'sft_data.jsonl'
if os.path.exists(sft_path):
    count = sum(1 for _ in open(sft_path))
    age   = (time.time() - os.path.getmtime(sft_path)) / 60
    print(f'✅ sft_data.jsonl: {count} examples ({age:.0f} min old) — reusing')
else:
    print('Generating SFT warmup data (~2 min)...')
    os.system('python sft_generator.py')
    count = sum(1 for _ in open(sft_path))
    print(f'✅ Generated {count} SFT examples')


## 🤖 Step 4 — Load Qwen2.5-1.5B (4-bit QLoRA)

In [ ]:
import os, sys, torch
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from unsloth import FastLanguageModel
from train_data_centric import load_model

model, tokenizer = load_model()

vram  = torch.cuda.memory_allocated() / 1e9
total = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f'\n📊 VRAM: {vram:.1f} GB used / {total:.1f} GB total ({100*vram/total:.0f}%)')


## 🎓 Step 5 — Phase 1: SFT Warmup

One epoch of supervised fine-tuning on ~9,480 heuristic trajectories.
Teaches the model valid command syntax before RL starts.
Without this, the model outputs gibberish and gets zero reward.

**TensorBoard:** Run Step 6 to open the live dashboard.


In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import run_sft_warmup

model = run_sft_warmup(model, tokenizer)
print('✅ SFT warmup complete — model knows command syntax')


## 📡 Step 6 — TensorBoard (Experiment Tracking)

Launch this now — it will show SFT logs immediately and GRPO logs as training progresses.
Refresh the TensorBoard tab every few minutes during Step 7.


In [ ]:
%load_ext tensorboard
%tensorboard --logdir /content/data-centric-env/logs


## 🏋️ Step 7 — Phase 2: GRPO Training

Agent learns *when* to use each command via reinforcement learning.
Each GRPO step: model generates a command → environment executes it → reward flows back.

**Reward discrimination:** The reward is now strictly discriminating:
- Hitting target efficiently (few steps): **+0.7 → +0.9**
- Hitting target slowly (many steps): **+0.3 → +0.5**
- Failing to hit target: **−0.1 → −0.5**
- Blind apply / blind submit: **large penalties**

Checkpoints saved every 25 steps to `./data-centric-checkpoints/`.
If the runtime disconnects, re-run all cells — it auto-resumes from the latest checkpoint.


In [ ]:
import os, sys, glob
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import run_grpo_training

# Auto-resume from latest checkpoint
ckpt_dir = './data-centric-checkpoints'
resume_from = None
if os.path.exists(ckpt_dir):
    checkpoints = sorted(glob.glob(f'{ckpt_dir}/checkpoint-*'))
    if checkpoints:
        resume_from = checkpoints[-1]
        print(f'Resuming from: {resume_from}')

model = run_grpo_training(model, tokenizer, resume_from_checkpoint=resume_from)
print('✅ GRPO training complete')


## 📈 Step 8 — Plot Reward Curves

Generates 4 plots from the training log:
- **reward_curve.png** — Episode reward over time with rolling mean
- **success_rate.png** — Success rate per curriculum level
- **accuracy_gain.png** — Accuracy gain vs baseline per episode
- **curriculum.png** — Curriculum level progression


In [ ]:
import os
os.chdir('/content/data-centric-env')

log_path = 'logs/training.jsonl'

if not os.path.exists(log_path):
    print('No training log yet — run Step 7 first')
else:
    n_eps = sum(1 for _ in open(log_path))
    print(f'Training log: {n_eps} episodes')

    os.system(f'python plot_rewards.py --log {log_path} --out plots/')

    from IPython.display import Image, display
    for fname in ['reward_curve.png', 'success_rate.png', 'accuracy_gain.png', 'curriculum.png']:
        path = f'plots/{fname}'
        if os.path.exists(path):
            print(f'\n--- {fname} ---')
            display(Image(path, width=900))


## 🏆 Step 9 — Evaluate: Trained Agent vs Baselines

Runs the trained agent against:
- **Random baseline** — random command each step
- **Heuristic baseline** — fixed rule-based sequence
- **Trained agent** — your GRPO-trained model


In [ ]:
import os, json
os.chdir('/content/data-centric-env')

os.system('python eval_data_centric.py')

if os.path.exists('eval_results.json'):
    with open('eval_results.json') as f:
        results = json.load(f)
    print('\n=== Evaluation Results ===')
    for k, v in results.items():
        print(f'  {k}: {v}')


## 💾 Step 10 — Save Trained LoRA Adapter

In [ ]:
import os, sys
os.chdir('/content/data-centric-env')
sys.path.insert(0, '/content/data-centric-env')

from train_data_centric import save_model
save_model(model, tokenizer)
print('✅ Adapter saved to ./data-centric-adapter/')

# List outputs
import glob
print('\n📁 Training outputs:')
for f in sorted(glob.glob('plots/*.png') + glob.glob('logs/*.jsonl') + ['eval_results.json']):
    if os.path.exists(f):
        size = os.path.getsize(f) / 1024
        print(f'  {f} ({size:.1f} KB)')


## 📤 Step 11 — Download Results

Download plots and training log to your local machine.


In [ ]:
from google.colab import files
import glob, os

to_download = glob.glob('plots/*.png') + ['logs/training.jsonl', 'eval_results.json']

for f in to_download:
    if os.path.exists(f):
        print(f'Downloading: {f}')
        files.download(f)
